<a href="https://colab.research.google.com/github/Arsh-Vora/Arxiv-PageRank-Analysis/blob/dev/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub -q

import kagglehub
import os

# Download latest version of the Arxiv dataset
print("Downloading Arxiv dataset...")
path = kagglehub.dataset_download("Cornell-University/arxiv")
json_path = os.path.join(path, "arxiv-metadata-oai-snapshot.json")

print("Path to dataset files:", json_path)

Using Colab cache for faster access to the 'arxiv' dataset.
Path to dataset files: /kaggle/input/arxiv/arxiv-metadata-oai-snapshot.json


### **Personal Log: Trying a faster way**
Okay, I'm starting with a totally different approach here. Instead of setting up a whole Spark cluster, I'm using `scipy.sparse` to build a direct author-paper matrix. Spark is great, but for a quick test, it feels like overkill.

I'm using the `authors_parsed` field to map everyone to a unique ID. If this works, it should be way faster than the RDD shuffles I was doing earlier.



In [ ]:

import json
import gc
import numpy as np
import scipy.sparse as sp
import time

print("Parsing JSON and building Author-Paper relationships...")
start_time = time.time()

author_to_id = {}
id_to_author = []

row_indices = []
col_indices = []

with open(json_path, 'r') as f:
    for paper_id, line in enumerate(f):
        record = json.loads(line)
        authors_parsed = record.get('authors_parsed', [])

        for author in authors_parsed:
            author_name = f"{author[0]}, {author[1]}"

            if author_name not in author_to_id:
                author_to_id[author_name] = len(id_to_author)
                id_to_author.append(author_name)

            row_indices.append(author_to_id[author_name])
            col_indices.append(paper_id)

n_authors = len(id_to_author)
n_papers = paper_id + 1

print(f"Extracted {n_authors} unique authors from {n_papers} papers.")

A = sp.csr_matrix((np.ones(len(row_indices), dtype=np.float32),
                   (row_indices, col_indices)),
                  shape=(n_authors, n_papers))

del row_indices, col_indices, author_to_id
gc.collect()

print(f"Data ingestion completed in {time.time() - start_time:.2f} seconds.")

Parsing JSON and building Author-Paper relationships...
Extracted 1965563 unique authors from 2975294 papers.
Data ingestion completed in 61.24 seconds.


### **Personal Log: The "Short Cut" Math**
This is the "magic" part. I'm implementing a low-rank approximation ($T^2$-Approx). Basically, instead of iterating a thousand times across the whole 23-million node graph, I'm shrinking the transition matrix down to a manageable size.

* **What I tried:** At first, I tried to do the full matrix multiplication, but Colab immediately ran out of RAM and the kernel restarted.
* **The Fix:** I switched to this approximation method. It’s much lighter on the memory. It feels a bit like "cheating" compared to the exact Spark math, but it finished in under 2 minutes.

In [ ]:
print("Constructing T^2-Approx matrices X and Y using degree-weighted sampling...")
step_time = time.time()

paper_degrees = A.T.dot(np.ones(n_authors, dtype=np.float32))
d = A.dot(paper_degrees)
d[d == 0] = 1.0

p_dist = d / np.sum(d)

c = 3000
np.random.seed(42)
S = np.random.choice(n_authors, c, replace=False, p=p_dist)

Y_unnormalized = A[S, :].dot(A.T)
inv_d_S = 1.0 / d[S]
Y = Y_unnormalized.multiply(inv_d_S[:, np.newaxis]).tocsr()

X_unnormalized = A.dot(A[S, :].T)
inv_d = 1.0 / d
X = X_unnormalized.multiply(inv_d[:, np.newaxis]).tocsr()

print(f"Matrices X and Y constructed in {time.time() - step_time:.2f} seconds.")

Constructing T^2-Approx matrices X and Y using degree-weighted sampling...
Matrices X and Y constructed in 8.66 seconds.


### **Personal Log: Fast results (but are they right?)**
Wow, 86 seconds total. That's insane compared to the hour-long Spark jobs. I'm sorting the top 20 now to see who "won."

* **Result Check:** The ATLAS Collaboration is #1. This makes sense because I didn't filter out any data here. However, some of the other names look a bit different from the Spark run.
* **Self-Correction:** This is good for a quick sanity check, but the Prof definitely wants to see the Spark version in the final report. I'll use this as a "baseline" to prove the Spark results are accurate.

In [ ]:
print("Performing Power Iteration on small matrix Z...")
step_time = time.time()

Z = Y.dot(X).toarray()

alpha = 0.85
tau = 1.0 / (100 * n_authors)

pi_0 = np.ones(n_authors, dtype=np.float32) / n_authors

pi_k = Y.dot(pi_0)
c_vector = np.ones(c, dtype=np.float32) / c

iterations = 0
for k in range(100):
    iterations += 1
    pi_next = (1 - alpha**2) * Z.dot(pi_k) + (alpha**2) * c_vector

    if np.linalg.norm(pi_next - pi_k, ord=2) < tau:
        break
    pi_k = pi_next

print(f"Converged after {iterations} iterations.")
print(f"Iteration completed in {time.time() - step_time:.2f} seconds.")

Performing Power Iteration on small matrix Z...
Converged after 10 iterations.
Iteration completed in 14.49 seconds.


### **My Notes: Running the math loop**
This is where the actual PageRank math happens. Instead of doing the full, massive calculation on the whole graph, I'm running "Power Iteration" on a much smaller matrix `Z`.

* **The Z Matrix:** This is basically a shrunken version of the graph. I multiplied `Y` and `X` to get it. It’s way smaller, which is why the loop runs so fast.
* **Alpha and Tau:** I used 0.85 for alpha because that’s the standard damping factor the prof mentioned. Tau is just the "stopping point"—once the numbers stop changing much, the code breaks out of the loop so it doesn't waste time.
* **Speed:** It converged in only 10 iterations! This whole part took less than 15 seconds, which is a massive relief after seeing how slow the Spark version was.

**Check:** The code broke out of the loop because it hit the `tau` threshold. This means it found the final scores (the stationary distribution) without needing to run all 100 times.

In [ ]:
print("Reconstructing PageRank vector and fetching top authors...")

pi_final = (1 - alpha) * pi_0 + (alpha / (1.0 + alpha)) * X.dot(pi_k)

pi_final = pi_final / np.sum(pi_final)

top_k = 20
top_indices = np.argsort(pi_final)[::-1][:top_k]

print(f"\n--- Top {top_k} Authors by Co-authorship PageRank ---")
for i, idx in enumerate(top_indices):
    print(f"{i+1}. {id_to_author[idx]} (Score: {pi_final[idx]:.8f})")

print(f"\nTotal Pipeline Execution Time: {time.time() - start_time:.2f} seconds.")

Reconstructing PageRank vector and fetching top authors...

--- Top 20 Authors by Co-authorship PageRank ---
1. ATLAS Collaboration,  (Score: 0.00009007)
2. Teubert, C. Matteuzzi. F. (Score: 0.00005797)
3. Kmiec, M . (Score: 0.00005786)
4. Sticchi, M. (Score: 0.00005589)
5. Lovell, G. (Score: 0.00005151)
6. Jans, P. Ilten E. (Score: 0.00005142)
7. Gang, R. (Score: 0.00005041)
8. Buoni, M. J. (Score: 0.00005033)
9. Shin, M. -W. (Score: 0.00004986)
10. Liang, Y. X. (Score: 0.00004967)
11. O'Hanlon, D. (Score: 0.00004916)
12. Manthey, J. (Score: 0.00004912)
13. Fontanna, M. (Score: 0.00004815)
14. Mohamed, H. (Score: 0.00004815)
15. Van Veghel, M. (Score: 0.00004815)
16. Guz, I. (Score: 0.00004815)
17. Linn, C. P. (Score: 0.00004815)
18. Schreiner, H. (Score: 0.00004756)
19. Carvalho, M. Féo Pereira Rivello (Score: 0.00004745)
20. Everett, C. (Score: 0.00004738)

Total Pipeline Execution Time: 84.69 seconds.


### **My Notes: Turning numbers back into names**
This is the final step for the matrix version. I'm basically taking the math results from the "pi_k" vector and mapping them back to the actual names of the 1.9 million authors.

* **The Math:** That `pi_final` formula looks pretty dense, but it's just the way to "fix" the approximation so it matches the real graph.
* **The Scores:** I divided by the sum of `pi_final` to make sure it's all normalized. This is why these scores are super small decimals (like 0.00009) while the Spark scores were in the hundreds. It’s just a different scale.
* **Sorting:** I used `argsort` to get the top 20. It's almost instant here because it's not distributed; it's all just sitting in my local memory.

**Total Time:** 84 seconds. That is crazy fast. Even though the prof wants the Spark version, it was definitely worth running this first to see that the ATLAS Collaboration really is the #1 hub in the data. It gave me a good "baseline" to know my Spark code was working right.

### **Comparison: Matrix vs. Spark**
1. **Speed:** Matrix (86s) beats Spark (90m) easily.
2. **Accuracy:** Matrix is an *approximation*. Spark is *exact*.
3. **Memory:** Matrix code is very "fragile"—if the dataset was any bigger, even the sparse matrix would probably crash. Spark is "robust" because it's designed to spill to disk if things get too heavy.

* **Final Decision:** I'll keep this in my project folder to show I explored multiple algorithms, but I'll focus the main report on the Spark Bipartite implementation.